# Koheron CTL200 — laser power

Turns the diode on or off over the Red Pitaya UART. This capability should be moved to the raspberry pi. This script does not load the FPGA or have anything to do with locking.

Change `rtset` (the temperature) any time without cycling the diode using the labeled cell below. Closing the serial port does not turn the laser off if you desire to do that.


In [ ]:
# ------------------------------------------------------------
# Setup — UART to laser only, no PyRPL / FPGA

# Best laser settings: 250 mA, 10390 - 10459 ohm, 0.001 P, 0.0005 I, 0.0001 D
# ------------------------------------------------------------

import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from laserlock.koheron import KoheronLaser, LaserSettings

HOSTNAME = "rp-f0c970.local"
SSH_USER = "root"
SSH_PASSWORD = "root"  # same as PyRPL; BatchMode/keys are not used
UART_PORT = "/dev/ttyPS1"

settings = LaserSettings(
    rtset_ohm=10420.0,
    pgain=0.001,
    igain=0.0005,
    dgain=0.0001,
    ilaser_ma=250.0,
    ilmax_ma=255.0,
    lckon=0,
    tprot=1,
    rtmin_ohm=10000.0,
    rtmax_ohm=11000.0,
    vtmin_v=-3.0,
    vtmax_v=3.0,
    warmup_s=300.0,
)

laser = KoheronLaser.connect(
    hostname=HOSTNAME,
    user=SSH_USER,
    password=SSH_PASSWORD,
    port=UART_PORT,
    settings=settings,
)
print("RP connected. Plug the Koheron UART before On / Status / RTSET.")


CTL200 UART /dev/ttyPS1 via ssh root@rp-f0c970.local
  RP ssh ok (rp-f0c970). UART /dev/ttyPS1 is not probed until On/Status.
RP connected. Plug the Koheron UART before On / Status / RTSET.


## Apply settings / RTSET

Writes PID, limits, and `rtset`. Does not change `lason`. Re-run after editing knobs in Setup, or use the next cell to change temperature only.


In [15]:
# ------------------------------------------------------------
# Apply PID + limits + rtset (diode state unchanged)
# ------------------------------------------------------------

laser.apply_settings()
laser.status()


Applied CTL200 settings: rtset 10420.000 Ω  PID 0.001/0.0005/0.0001  ilaser 250 mA  ilmax 255 mA
lason 0  ilaser 250.000 mA  vlaser -0.00026 V  rtset 10420.000 Ω  rtact 9998.806 Ω  tecon 1  err c


{'lason': 0,
 'ilaser': 250.0,
 'vlaser': -0.00026,
 'rtset': 10420.0,
 'rtact': 9998.806,
 'tecon': 1,
 'err': 'c',
 'raw': {'lason': '0',
  'ilaser': '250.000',
  'vlaser': '-0.00026',
  'rtset': '10420.000',
  'rtact': '9998.806',
  'tecon': '1',
  'err': 'c'}}

## Gives you the oppurtunity to set the temp after laser is on/initialized

In [ ]:

#RTSET_OHM = 10440.0
RTSET_OHM = 10415.0

laser.set_rtset(RTSET_OHM)
laser.status()


rtset -> 10415.000 Ω
lason 1  ilaser 250.000 mA  vlaser 1.85924 V  rtset 10415.000 Ω  rtact 10413.755 Ω  tecon 1  err c


{'lason': 1,
 'ilaser': 250.0,
 'vlaser': 1.85924,
 'rtset': 10415.0,
 'rtact': 10413.755,
 'tecon': 1,
 'err': 'c',
 'raw': {'lason': '1',
  'ilaser': '250.000',
  'vlaser': '1.85924',
  'rtset': '10415.000',
  'rtact': '10413.755',
  'tecon': '1',
  'err': 'c'}}

## Laser On

Applies settings, enables TEC, enables diode current. Returns immediately. Use the other notebook now. 


In [17]:
# ------------------------------------------------------------
# On — no warmup sleep
# ------------------------------------------------------------

laser.on()
laser.status()


Applied CTL200 settings: rtset 10405.000 Ω  PID 0.001/0.0005/0.0001  ilaser 250 mA  ilmax 255 mA
Laser ON  ilaser setpoint 250 mA  readback 250.000 mA  err c
No warmup wait. Call wait_stable() only if you want to block.
lason 1  ilaser 250.000 mA  vlaser 1.85882 V  rtset 10405.000 Ω  rtact 10359.990 Ω  tecon 1  err c


{'lason': 1,
 'ilaser': 250.0,
 'vlaser': 1.85882,
 'rtset': 10405.0,
 'rtact': 10359.99,
 'tecon': 1,
 'err': 'c',
 'raw': {'lason': '1',
  'ilaser': '250.000',
  'vlaser': '1.85882',
  'rtset': '10405.000',
  'rtact': '10359.990',
  'tecon': '1',
  'err': 'c'}}

## Wait (I recommend waiting 6-7 minutes for laser stability)

In [ ]:
# # ------------------------------------------------------------
# # Optional settle — skip this cell during testing
# # ------------------------------------------------------------

# WARMUP_S = 360.0

# laser.wait_stable(WARMUP_S)


In [19]:
# ------------------------------------------------------------
# Status
# ------------------------------------------------------------

laser.status()


lason 1  ilaser 250.000 mA  vlaser 1.85890 V  rtset 10405.000 Ω  rtact 10401.159 Ω  tecon 1  err c


{'lason': 1,
 'ilaser': 250.0,
 'vlaser': 1.8589,
 'rtset': 10405.0,
 'rtact': 10401.159,
 'tecon': 1,
 'err': 'c',
 'raw': {'lason': '1',
  'ilaser': '250.000',
  'vlaser': '1.85890',
  'rtset': '10405.000',
  'rtact': '10401.159',
  'tecon': '1',
  'err': 'c'}}

## Turn laser off

In [ ]:


# laser.off()
# laser.status()


Laser OFF (TEC still on)
lason 0  ilaser 250.000 mA  vlaser 0.33245 V  rtset 10415.000 Ω  rtact 10335.398 Ω  tecon 1  err c


{'lason': 0,
 'ilaser': 250.0,
 'vlaser': 0.33245,
 'rtset': 10415.0,
 'rtact': 10335.398,
 'tecon': 1,
 'err': 'c',
 'raw': {'lason': '0',
  'ilaser': '250.000',
  'vlaser': '0.33245',
  'rtset': '10415.000',
  'rtact': '10335.398',
  'tecon': '1',
  'err': 'c'}}

In [ ]:
# # ------------------------------------------------------------
# # Close UART — does not change lason
# # ------------------------------------------------------------

# laser.close()


CTL200 UART released (diode state unchanged).
